In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(base_url='https://openrouter.ai/api/v1',api_key=os.environ.get('OPENAI_API_KEY'),model="gpt-3.5-turbo",temperature=0)
#OpenAI(base_url="https://openrouter.ai/api/v1",api_key=os.environ.get('OPENAI_API_KEY'))

## **RAG IMPLEMENTATION WITH OWN TEXT DATA** ##

#### **STEP 1: Preparing Document for our text** ####

In [ ]:
from langchain_core.documents import Document

In [ ]:
my_text="""Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]"""

In [ ]:
docs=[Document(page_content=my_text,metadata={"source":"ABC","documentID":"Doc1"})]
docs

#### **STEP 2: Splitting the document into CHUNKS** ####

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks = splitter.split_documents(docs)
chunks

#### **STEP 3: Creating Embeddings for the CHUNKS** ####

In [ ]:
from langchain_openai import OpenAIEmbeddings 

In [ ]:
embedding_model = OpenAIEmbeddings(
    openai_api_base="https://openrouter.ai/api/v1",
    model="openai/text-embedding-3-small"
)

#### **Step4: Create and Store Embeddings of the chunks in Vector DB** ####

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=chunks,embedding=embedding_model)

#### **Step5: Semantic Search** ####

In [ ]:
vectors = []
for doc in chunks:
    vector = embedding_model.embed_documents([doc.page_content])
    vectors.append(vector)


In [ ]:
context=vectorstore.similarity_search('What is AI?',k=2)

#### **Step5: Talk to LLM** ####

In [ ]:
llm.invoke('What is AI? Answer using following context:{context}')